# Assignment 6: Machine Translation System

## Objective
Develop a Machine Translation system to translate public information content between English and Hindi (Indian Language).

## 1. Install Required Libraries

In [ ]:
%pip install transformers sentencepiece sacremoses torch sacrebleu
print("Libraries installed successfully!")

## 2. Import Libraries

In [ ]:
from transformers import MarianMTModel, MarianTokenizer, pipeline
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## 3. Load Translation Models

Loading pre-trained Helsinki-NLP models for English-Hindi and Hindi-English translation.

In [ ]:
# English to Hindi Model
model_name_en_hi = "Helsinki-NLP/opus-mt-en-hi"
tokenizer_en_hi = MarianTokenizer.from_pretrained(model_name_en_hi)
model_en_hi = MarianMTModel.from_pretrained(model_name_en_hi)

print("English to Hindi model loaded successfully!")
print(f"Model: {model_name_en_hi}")

In [ ]:
# Hindi to English Model
model_name_hi_en = "Helsinki-NLP/opus-mt-hi-en"
tokenizer_hi_en = MarianTokenizer.from_pretrained(model_name_hi_en)
model_hi_en = MarianMTModel.from_pretrained(model_name_hi_en)

print("Hindi to English model loaded successfully!")
print(f"Model: {model_name_hi_en}")

## 4. Define Sample Public Information Text

Creating sample English text containing public information for translation.

In [ ]:
# Sample English sentences (Public Information)
english_texts = [
    "The government has announced a new health insurance scheme for citizens.",
    "Public transportation services will be available from 6 AM to 11 PM daily.",
    "All citizens must carry valid identification documents when traveling.",
    "The vaccination drive will begin next Monday at all government hospitals.",
    "Emergency services can be reached by dialing 112 from any phone."
]

print("Sample Public Information Texts:")
print("=" * 70)
for i, text in enumerate(english_texts, 1):
    print(f"{i}. {text}")

## 5. Translate English to Hindi

Translating public information from English to Hindi.

In [ ]:
def translate_en_to_hi(text):
    """Translate English text to Hindi"""
    inputs = tokenizer_en_hi(text, return_tensors="pt", padding=True, truncation=True)
    translated = model_en_hi.generate(**inputs)
    translated_text = tokenizer_en_hi.decode(translated[0], skip_special_tokens=True)
    return translated_text

# Translate all English texts to Hindi
hindi_translations = []
print("English to Hindi Translations:")
print("=" * 70)

for i, eng_text in enumerate(english_texts, 1):
    hindi_text = translate_en_to_hi(eng_text)
    hindi_translations.append(hindi_text)
    print(f"\n{i}. English: {eng_text}")
    print(f"   Hindi: {hindi_text}")

## 6. Translate Hindi to English

Performing reverse translation from Hindi back to English.

In [ ]:
def translate_hi_to_en(text):
    """Translate Hindi text to English"""
    inputs = tokenizer_hi_en(text, return_tensors="pt", padding=True, truncation=True)
    translated = model_hi_en.generate(**inputs)
    translated_text = tokenizer_hi_en.decode(translated[0], skip_special_tokens=True)
    return translated_text

# Translate Hindi back to English
back_translations = []
print("Hindi to English (Reverse) Translations:")
print("=" * 70)

for i, hindi_text in enumerate(hindi_translations, 1):
    back_trans = translate_hi_to_en(hindi_text)
    back_translations.append(back_trans)
    print(f"\n{i}. Hindi: {hindi_text}")
    print(f"   Back-translated English: {back_trans}")
    print(f"   Original English: {english_texts[i-1]}")

## 7. Evaluate Translation Quality (BLEU Score)

BLEU (Bilingual Evaluation Understudy) score measures translation quality.

In [ ]:
from sacrebleu.metrics import BLEU

# Calculate BLEU score for back-translation
bleu = BLEU()

# BLEU score comparison
print("Translation Quality Evaluation:")
print("=" * 70)

individual_scores = []
for i, (original, back_trans) in enumerate(zip(english_texts, back_translations), 1):
    # BLEU score expects references as list of strings
    score = bleu.sentence_score(back_trans, [original])
    individual_scores.append(score.score)
    print(f"\n{i}. BLEU Score: {score.score:.2f}")
    print(f"   Original: {original}")
    print(f"   Back-translated: {back_trans}")

# Overall BLEU score
overall_score = bleu.corpus_score(back_translations, [english_texts])
print(f"\n{'=' * 70}")
print(f"Overall Corpus BLEU Score: {overall_score.score:.2f}")
print(f"Average Individual BLEU Score: {sum(individual_scores)/len(individual_scores):.2f}")

## 8. Test with Multiple Sentences

Testing with various types of public information content.

In [ ]:
# More comprehensive test cases
test_sentences = {
    "Health": "Free COVID-19 testing is available at all primary health centers.",
    "Education": "The new academic session will begin on July 1st for all schools.",
    "Transportation": "Metro services are suspended on Republic Day for security reasons.",
    "Legal": "Citizens have the right to information under RTI Act 2005.",
    "Safety": "Please follow traffic rules and wear helmets for your safety."
}

print("Comprehensive Translation Test:")
print("=" * 80)

results = []
for category, sentence in test_sentences.items():
    hindi = translate_en_to_hi(sentence)
    back_eng = translate_hi_to_en(hindi)
    bleu_score = bleu.sentence_score(back_eng, [sentence]).score
    
    results.append({
        'Category': category,
        'English': sentence,
        'Hindi': hindi,
        'Back-Translation': back_eng,
        'BLEU Score': f"{bleu_score:.2f}"
    })
    
    print(f"\n{category}:")
    print(f"  EN: {sentence}")
    print(f"  HI: {hindi}")
    print(f"  Back: {back_eng}")
    print(f"  BLEU: {bleu_score:.2f}")

# Display as DataFrame
df_results = pd.DataFrame(results)
print("\n" + "=" * 80)
print("Summary Table:")
print(df_results.to_string(index=False))